In [2]:
import run
import chephy_model as cpm
import numpy as np
import torch
from torch_geometric.data import HeteroData
from torch_geometric.transforms import ToUndirected
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
def get_epitope_features(epitopes, atchley_dict):
    # Create epitope features (mean of Atchley vectors per sequence)
    epitope_features = []
    for epitope in epitopes:
        vecs = [list(atchley_dict[aa].values()) for aa in epitope if aa in atchley_dict]
        if vecs:
            mean_vec = np.mean(vecs, axis=0)
        else:
            mean_vec = np.zeros(len(next(iter(atchley_dict.values()))))
        epitope_features.append(mean_vec)

    epitope_features = torch.tensor(epitope_features, dtype=torch.float)
    return epitope_features

In [4]:
def get_tcr_features(sequences_list, atchley_dict):
    # Use Atchley vectors as node features for TCRs
    tcr_features = []
    for seq in sequences_list:
        vecs = [list(atchley_dict[aa].values()) for aa in seq if aa in atchley_dict]
        tcr_features.append(np.concatenate(vecs))
    tcr_features = torch.tensor(tcr_features, dtype=torch.float)  # [num_tcr, 8*5]
    return tcr_features

In [5]:
def build_hetero_graph(data, chephy_matrix, hamming_matrix, sequences_list, epitopes):
    # Load Atchley dictionary
    atchley_df = pd.read_csv("/home/dsi/chenbis/repos/sol_lab/files/atchley.csv")
    atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')    

    epitope_features = get_epitope_features(epitopes, atchley_dict)
    tcr_features = get_tcr_features(sequences_list, atchley_dict)

    # Get edges from the data
    edge_index = torch.tensor(data[['tcr_index', 'epitope_index']].dropna().astype(int).values.T, dtype=torch.long)  # shape [2, num_edges]
    
    # Convert chephy_matrix (distance) into edge index for TCR similarity
    threshold = 1  # You can tune this value

    rows, cols = np.where((chephy_matrix < threshold) & (chephy_matrix > 0))  # Exclude self-loops
    edge_pairs = list(zip(rows, cols))    
    
    tcr_sim_edge_index = torch.tensor(edge_pairs, dtype=torch.long).T  # shape [2, num_edges]

    # Similarity = 1 - distance
    edge_weights = 1 - chephy_matrix[rows, cols]
    edge_weights = torch.tensor(edge_weights, dtype=torch.float)
    edge_scores = torch.tensor(data['vdjdb.score'].values, dtype=torch.float)
        
    data_hetero = HeteroData()

    data_hetero['tcr'].x = tcr_features
    data_hetero['epitope'].x = epitope_features
    data_hetero['tcr', 'binds', 'epitope'].edge_index = edge_index
    data_hetero['tcr', 'binds', 'epitope'].edge_attr = edge_scores
    data_hetero['tcr', 'similar', 'tcr'].edge_index = tcr_sim_edge_index
    data_hetero['tcr', 'similar', 'tcr'].edge_attr = edge_weights


    num_tcr = len(sequences_list)
    num_epitopes = len(epitopes)
    # [num_tcr, num_epitopes], multi-label format
    tcr_labels = torch.zeros((num_tcr, num_epitopes), dtype=torch.float)

    for row in data.itertuples():
        tcr_idx = row.tcr_index
        epi_idx = row.epitope_index
        if tcr_idx != -1 and epi_idx != -1:
            tcr_labels[tcr_idx, epi_idx] = 1.0

    data_hetero['tcr'].y = tcr_labels  # shape: [num_tcr, num_epitopes]


    # create masks for training, validation and testing
    valid_indices = (tcr_labels != -1).nonzero(as_tuple=True)[0]
    train_idx, temp_idx = train_test_split(valid_indices, test_size=0.3, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    num_tcr = data_hetero['tcr'].num_nodes
    train_mask = torch.zeros(num_tcr, dtype=torch.bool)
    val_mask = torch.zeros(num_tcr, dtype=torch.bool)
    test_mask = torch.zeros(num_tcr, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    data_hetero['tcr'].train_mask = train_mask
    data_hetero['tcr'].val_mask = val_mask
    data_hetero['tcr'].test_mask = test_mask

    return data_hetero


In [6]:
input_file = "../files/vdjdb_score3.csv"
max_dist=1
label_header = "antigen.epitope"
cdr3_header = "cdr3"

data = run.load_dataframe(input_file)

# data = read_csv_files_from_folder()

data = run.prepare_data(data, cdr3_header, label_header)    
data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

sequences = set(data["cdr3_truncated"])
epitopes = list(set(data["antigen.epitope"]))

chephy_matrix, hamming_matrix, sequences_list = cpm.find_che_phy_dist(sequences)

data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
data['epitope_index'] = data['antigen.epitope'].apply(lambda x: epitopes.index(x) if x in epitopes else -1)

sequence_indices = np.arange(len(sequences_list))


In [7]:
sequences_list.index("YSIRGSRG")

512

In [8]:
data_hetero = build_hetero_graph(data, chephy_matrix, hamming_matrix, sequences_list, epitopes)


/tmp/ipykernel_1385589/3346956718.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647327489/work/torch/csrc/utils/tensor_new.cpp:278.)
  epitope_features = torch.tensor(epitope_features, dtype=torch.float)
